In [22]:
import sklearn.metrics as metrics
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split, KFold
from sklearn import preprocessing, svm
from evaluacion_funciones import *

In [23]:
# Carga de datos.
datos = load_iris()
X = datos.data
y = datos.target
print(np.shape(X))

(150, 4)


In [24]:
# Métricas de evaluación.
metricas = {
  'ACC':    metrics.accuracy_score,
  'PREC':   lambda y_true, y_pred:
            metrics.precision_score(y_true, y_pred,
              average='micro'),
  'RECALL': lambda y_true, y_pred:
            metrics.recall_score(y_true, y_pred,
              average='micro'),
  'F1':     lambda y_true, y_pred:
            metrics.f1_score(y_true, y_pred, average='micro')}

In [25]:
# 1) Partición de datos externa
X_training, X_testing, y_training, y_testing = train_test_split(X, y, test_size=0.2, random_state=42)
print(np.shape(X_training))

(120, 4)


In [ ]:
# ------------- TRAINING ---------------- #

In [26]:
# 2) Extracción de características
from sklearn.feature_selection import SelectPercentile


scaler = preprocessing.StandardScaler()
X_train_stdr = scaler.fit_transform(X_training) # Estandarización de los datos de entrenamiento
X_test_stdr = scaler.transform(X_testing) # Estandarización de los datos de test

# 3) Selección de atributos
selector = SelectPercentile(percentile=90) # Selección del 90% de los mejores atributos
X_train_selected = selector.fit_transform(X_train_stdr, y_training) # Selección de los atributos de entrenamiento
X_test_selected = selector.transform(X_test_stdr) # Selección de los atributos de test
print(np.shape(X_train_selected)) # Comprobación de la nueva dimensión de los datos de entrenamiento

(120, 3)


In [27]:
# 4) Estandarización de los datos de entrenamiento
standardizer = preprocessing.StandardScaler()
stdr_trained = standardizer.fit(X_training)
X_stdr = stdr_trained.transform(X_training)
print(X_stdr)

[[-1.47393679  1.20365799 -1.56253475 -1.31260282]
 [-0.13307079  2.99237573 -1.27600637 -1.04563275]
 [ 1.08589829  0.08570939  0.38585821  0.28921757]
 [-1.23014297  0.75647855 -1.2187007  -1.31260282]
 [-1.7177306   0.30929911 -1.39061772 -1.31260282]
 [ 0.59831066 -1.25582892  0.72969227  0.95664273]
 [ 0.72020757  0.30929911  0.44316389  0.4227026 ]
 [-0.74255534  0.98006827 -1.27600637 -1.31260282]
 [-0.98634915  1.20365799 -1.33331205 -1.31260282]
 [-0.74255534  2.32160658 -1.27600637 -1.44608785]
 [-0.01117388 -0.80864948  0.78699794  0.95664273]
 [ 0.23261993  0.75647855  0.44316389  0.55618763]
 [ 1.08589829  0.08570939  0.55777524  0.4227026 ]
 [-0.49876152  1.87442714 -1.39061772 -1.04563275]
 [-0.49876152  1.4272477  -1.27600637 -1.31260282]
 [-0.37686461 -1.47941864 -0.01528151 -0.24472256]
 [ 0.59831066 -0.58505976  0.78699794  0.4227026 ]
 [ 0.72020757  0.08570939  1.01622064  0.8231577 ]
 [ 0.96400139 -0.13788033  0.38585821  0.28921757]
 [ 1.69538284  1.20365799  1.36

In [28]:
# 5) Construcción del algoritmo de aprendizaje.
# SVM con C=10, random_state=42 y probability=True  y Logistic Regression con C=10, random_state=42 y max_iter=1000
# C es el parámetro de regularización, que controla el equilibrio entre maximizar el margen y minimizar el error de clasificación.

algoritmos = {'SVM': svm.SVC(C=10, random_state=42, probability=True), 'LR': LogisticRegression(C=10, random_state=42, max_iter=1000)}


In [30]:
# 5.1) Validación cruzada interna y Optimización de los hiperparámetros
y_pred = {}
print("Algoritmos a evaluar:", algoritmos.keys())
for nombre, alg in algoritmos.items():
    y_pred[nombre] = cross_val_predict(alg, X_train_selected, y_training, cv=KFold(n_splits=10, shuffle=True, random_state=42))
    results = evaluacion(y_training, y_pred[nombre], metricas)
    print(f"Resultados para {nombre}:")
    for metrica, valor in results.items():
        print(f"  {metrica}: {valor:.4f}")
    print("Matriz de confusión:")
    print(metrics.confusion_matrix(y_training, y_pred[nombre]))
    results = cross_val_score(alg, X_train_selected, y_training, cv = KFold(n_splits=10, shuffle=True, random_state=42))
    print("Accuracy:   %0.4f +/- %0.4f" % (results.mean(), results.std()))


Algoritmos a evaluar: dict_keys(['SVM', 'LR'])
Resultados para SVM:
  ACC: 0.9417
  PREC: 0.9417
  RECALL: 0.9417
  F1: 0.9417
Matriz de confusión:
[[40  0  0]
 [ 0 38  3]
 [ 0  4 35]]
Accuracy:   0.9417 +/- 0.0750
Resultados para LR:
  ACC: 0.9500
  PREC: 0.9500
  RECALL: 0.9500
  F1: 0.9500
Matriz de confusión:
[[40  0  0]
 [ 0 38  3]
 [ 0  3 36]]
Accuracy:   0.9500 +/- 0.0667


In [17]:
# 5.2) Entrenamiento del modelo definitivo
model = algoritmos['SVM'].fit(X_train_selected, y_training)

In [ ]:
# ------------- PREDICTION ---------------- #

In [18]:
# 6) Extracción de las características de test
X_test_stdr = stdr_trained.transform(X_testing)
X_test_selected = selector.transform(X_test_stdr)

# 7) Selección de los atributos de test
X_test_selected = selector.transform(X_test_stdr)

In [19]:
# 8) Estandarización de las característiacs de test
X_test_stdr = stdr_trained.transform(X_testing)

In [ ]:
# 9) Predicción del conjunto de test
y_pred_test = model.predict(X_test_selected)
print(y_pred_test)

[1 0 2 1 1 0 1 2 2 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


In [21]:
# 10) Evaluación del modelo sobre el conjunto de test
results = evaluacion(y_testing, y_pred_test, metricas)
print(results)
print(metrics.confusion_matrix(y_testing, y_pred_test))

{'ACC': 0.9666666666666667, 'PREC': 0.9666666666666667, 'RECALL': 0.9666666666666667, 'F1': 0.9666666666666667}
[[10  0  0]
 [ 0  8  1]
 [ 0  0 11]]


In [ ]:
# Ploteamos la curva ROC
y_proba_test = model.predict_proba(X_test_selected) # "predict_proba" para extraer probabilidades vez de predicciones

y_test_bin = preprocessing.label_binarize(y_testing, classes=[0,1,2]) # Usar "label_binarize" en el caso de problemas multiclase

auc = metrics.roc_auc_score(y_testing, y_proba_test, multi_class='ovr') # Area Under the ROC curve (AUC)

fpr, tpr, th = metrics.roc_curve(y_test_bin[:,0], y_proba_test[:,0])

plt.plot(fpr, tpr)
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('AUC = ' + str(np.round(auc,4)))
plt.grid()
plt.show()